# 02 — Create a Bedrock Managed Knowledge Base with Web Crawler

This notebook creates a BMKB using the **Web Crawler** connector to crawl websites and index their content.

**What "Bedrock Managed" means:** Bedrock handles the vector store for you. The Web Crawler connector crawls seed URLs, follows links, and indexes page content — no S3 upload needed.

### What this notebook does

1. Configures your environment (region, seed URLs, crawl settings)
2. Creates the BMKB with IAM roles and policies (auto-managed by utility)
3. Creates a Web Crawler data source with configurable depth, scope, and rate limits
4. Runs ingestion (crawls the websites)
5. Queries the KB using Retrieve and AgenticRetrieveStream
6. Cleans up all resources

### Prerequisites

- AWS credentials with Bedrock and IAM permissions
- Enable model access for your embedding and generation models
- For authenticated sites: a Secrets Manager secret with credentials

### Architecture

```
Seed URLs ──► Web Crawler ──► BMKB (Bedrock-managed vector store) ──► Retrieve / AgenticRetrieveStream
                  │                    │
                  ├── Crawl depth      ├── IAM Role (auto-created)
                  ├── Rate limits      ├── Embedding: Managed default (no extra cost)
                  └── Scope control    └── Parsing: Smart Parsing
```

### Supported Authentication

| Auth Type | Description |
|-----------|-------------|
| `NO_AUTH` | Public websites (default) |
| `BASIC_AUTH` | Username/password via Secrets Manager |
| `FORM` | Form-based login with XPath selectors |
| `SAML` | SAML SSO with XPath selectors |

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

Update the values below to match your environment.

In [ ]:
import boto3
import sys
import time
import logging
import pprint

sys.path.insert(0, "../..")

# Clients
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# Generate unique suffix
suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration (update these) ─────────────────────────────────────
knowledge_base_name = f'bmkb-webcrawler-{suffix}'
knowledge_base_description = 'BMKB workshop - Web Crawler data source'

# Seed URLs to crawl (max 10)
seed_urls = [
    'https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html'
]

# Optional: sitemap URLs (max 3)
sitemap_urls = None  # e.g. ['https://docs.example.com/sitemap.xml']

# Authentication: NO_AUTH | BASIC_AUTH | FORM | SAML
auth_type = 'NO_AUTH'
secret_arn = None  # Required if auth_type != NO_AUTH

# Crawl settings
crawl_depth = 2          # 0-10 (default 2)
max_links_per_url = 100  # 1-1000 (default 100)
max_crawled_urls_per_minute = 300  # 1-1200 (default 300)
sync_scope = 'ALL_DOMAINS'  # SUB_DOMAINS or ALL_DOMAINS
crawl_attachments = True

# Models
# Embedding model — use None for managed default (no extra cost)
# Or specify a custom Bedrock model (additional cost applies)
embedding_model = None  # Managed default
# embedding_model = 'amazon.titan-embed-text-v2:0'  # Custom — additional cost
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Seed URLs:  {seed_urls}')
print(f'Auth:       {auth_type}')
print(f'Depth:      {crawl_depth}')
print(f'Scope:      {sync_scope}')
print(f'Embedding:  {embedding_model}')
print(f'Generation: {generation_model_arn}')

## Step 2 — Create the Bedrock Managed Knowledge Base with Web Crawler

The unified `ManagedKnowledgeBase` utility handles everything:
- Creates IAM execution role with scoped policies (model + CloudWatch + optional Secrets Manager)
- Creates a `MANAGED` type KB
- Creates a Web Crawler data source with your crawl configuration via the `data_sources` parameter
- Waits for all resources to become active

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    data_sources=[{
        'type': 'WEB',
        'seed_urls': seed_urls,
        'sitemap_urls': sitemap_urls,
        'auth_type': auth_type,
        'secret_arn': secret_arn,
        'crawl_depth': crawl_depth,
        'max_links_per_url': max_links_per_url,
        'max_crawled_urls_per_minute': max_crawled_urls_per_minute,
        'sync_scope': sync_scope,
        'crawl_attachments': crawl_attachments,
    }],
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 3 — Ingest (Crawl websites)

Start an ingestion job. The Web Crawler will follow links from your seed URLs up to the configured depth and index the content.

In [ ]:
# ensure KB is ready
time.sleep(30)
job = kb.start_ingestion_job()

## Step 4 — Query the Knowledge Base

### 4a. Retrieve API — Raw Chunks

In [ ]:
response = kb.retrieve('What is Amazon Bedrock Knowledge Bases?', num_results=5)

print('=== Retrieve API ===')
for i, res in enumerate(response.get('retrievalResults', []), 1):
    score = res['score']
    text = res['content']['text'][:120]
    uri = res.get('location', {}).get('webLocation', {}).get('url', '')
    print(f'{i}. score={score:.4f} | {text}...')
    print(f'   source: {uri}')
print(f'\nTotal: {len(response.get("retrievalResults", []))} chunks')

### 4b. AgenticRetrieveStream (with generation)

In [ ]:
result = kb.agentic_retrieve_stream(
    query='How do I create a knowledge base in Amazon Bedrock?',
    model_arn=generation_model_arn,
    generate_response=True,
)

print('=== AgenticRetrieveStream (with generation) ===')
print(f"Answer:\n{result['generated_response']['answer']}")
print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")
for i, citation in enumerate(result['generated_response'].get('citations', [])[:3], 1):
    refs = citation.get('references', [])
    start = citation.get('startIndex', 0)
    end = citation.get('endIndex', 0)
    print(f'  Citation {i} [chars {start}-{end}]: {len(refs)} reference(s)')
    for ref in refs[:2]:
        idx = ref.get('resultIndex', -1)
        if idx >= 0 and idx < len(result['results']):
            print(f'    - result[{idx}]: {result["results"][idx]["content"]["text"][:80]}...')


### 4c. AgenticRetrieveStream API — Query Decomposition

In [ ]:
result = kb.agentic_retrieve_stream(
    query='What are the different types of knowledge bases and how do they compare?',
    model_arn=generation_model_arn,
    max_results=10,
    max_iterations=3,
)

print('=== AgenticRetrieveStream API ===')
print(f'Trace events: {len(result["traces"])}')
for t in result['traces']:
    attrs = t.get('attributes', {})
    print(f'  [{attrs.get("step", "?")}] {attrs.get("status", "")}')

print(f'\nFinal: {len(result["results"])} deduplicated chunks')
for i, r in enumerate(result['results'][:5], 1):
    text = r['content']['text'][:120]
    print(f'  {i}. {text}...')
# Show generated response if available
if result.get('generated_response'):
    print(f"\nGenerated Answer:\n{result['generated_response']['answer']}")
    print(f"Citations: {len(result['generated_response'].get('citations', []))}")


### 4d. Try your own queries

In [ ]:
QUERY = 'What embedding models are supported?'

result = kb.agentic_retrieve_stream(
    query=QUERY,
    model_arn=generation_model_arn,
    generate_response=True,
)
print(result['generated_response']['answer'])


## Step 5 — Add more data sources (optional)

You can add additional data sources to the same KB. For example, add an S3 source alongside the Web Crawler:

In [ ]:
# Uncomment to add an S3 data source to this KB
# s3_ds_id = kb.add_data_source({
#     'type': 'S3',
#     'bucket_name': f'bedrock-bmkb-docs-{account_id}',
#     's3_prefix': 'documents/',
# })
# print(f'Added S3 data source: {s3_ds_id}')
#
# # Ingest the new data source
# kb.start_ingestion_job(ds_id=s3_ds_id)
#
# # View all data sources
# print(f'Data sources: {kb.data_sources}')

## Step 6 — Cleanup

Delete all resources. The utility handles correct ordering:
data sources → KB → IAM roles/policies.

> Only run this when you're done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting Knowledge Base and associated resources==============================')
# kb.delete_kb(delete_iam=True)

In [ ]:
# Alternative: delete by KB ID only
# from utils.managed_knowledge_base import ManagedKnowledgeBase
# ManagedKnowledgeBase.delete_kb_by_id('YOUR_KB_ID', region_name='us-west-2')

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED` — Bedrock handles the vector store |
| Data Source | Web Crawler via `MANAGED_KNOWLEDGE_BASE_CONNECTOR` |
| Parsing | Smart Parsing (default, managed by Bedrock) |
| Embedding | Managed default (no extra cost) or custom |
| IAM | Auto-created scoped role + policies (model, CloudWatch, optional Secrets Manager) |
| Retrieval | `Retrieve` (raw chunks), `AgenticRetrieveStream` (agentic retrieval with optional generation) |

### Web Crawler Configuration Options

| Parameter | Default | Range | Description |
|---|---|---|---|
| `crawl_depth` | 2 | 0-10 | How deep to follow links |
| `max_links_per_url` | 100 | 1-1000 | Max links to follow per page |
| `max_crawled_urls_per_minute` | 300 | 1-1200 | Rate limit |
| `sync_scope` | ALL_DOMAINS | SUB_DOMAINS / ALL_DOMAINS | Which domains to crawl |
| `crawl_attachments` | True | True / False | Whether to crawl linked attachments |
| `auth_type` | NO_AUTH | NO_AUTH / BASIC_AUTH / FORM / SAML | Authentication method |